In [1]:
import os
import optuna
import pandas as pd
from optuna.trial import TrialState
from optuna.study import StudyDirection

In [2]:
pd.set_option('display.precision', 25)

In [4]:
import os
import pandas as pd
import optuna
from optuna.trial import TrialState

def best_trial(
    db_path: str,
    study_name: str,
    n_trials: int = 50,
    direction: str = 'min'
):
    study = optuna.load_study(
        study_name=study_name,
        storage=f"sqlite:///{db_path}"
    )

    first_n = sorted(study.trials, key=lambda t: t.number)[:n_trials]
    completed = [t for t in first_n if t.state == TrialState.COMPLETE]

    if not completed:
        raise ValueError("No completed trials found in the first N trials.")

    if direction == 'min':
        best = min(completed, key=lambda t: t.value)
    elif direction == 'max':
        best = max(completed, key=lambda t: t.value)
    else:
        raise ValueError("Direction must be 'min' or 'max'")

    return {
        'trial_number': best.number,
        'value': best.value,
        'learning_rate': best.params.get('learning_rate'),
        'weight_decay': best.params.get('weight_decay'),
        'pooling': best.params.get('pooling'),
        'use_numeric': best.params.get('use_numeric'),
        'dropout':best.params.get('dropout')
    }

# Loop and collect results
dbs_path = '/scratch/sas10092/ehr-foundation/models/optuna_dbs/'
dbs = os.listdir(dbs_path)
dbs.remove('without_retrieval')
dbs.remove('with_retrieval')
rows = []
for db in dbs:
    arch = db.split('_')[0]
    if arch == 'big':
        arch = 'big_bird'
    task = db[len(arch)+1:-3]

    result = best_trial(
        db_path=os.path.join(dbs_path, db),
        study_name=arch,
        n_trials=25,
        direction='min'
    )
    row = {
        'arch': arch,
        'task': task,
        **result
    }
    rows.append(row)

# Convert to DataFrame and reorder columns
df = pd.DataFrame(rows)
# df = df[['arch', 'task', 'trial_number', 'value', 'learning_rate', 'weight_decay', 'pooling', 'use_numeric','dropout']]

# Optionally save to CSV


# Show result
df = df.sort_values(['task'])
# df = df[df.arch != 'bert']
df = df.reset_index(drop=True)
# df.to_csv("best_trials_summary.csv", index=False)

In [6]:
df.sort_values(by='arch')#[df.dropout.isna()==False]#.sort_values('arch')

,arch,task,trial_number,value,learning_rate,weight_decay,pooling,use_numeric,dropout
0,bert,behrt_y_mort_1yr,10,0.3340676128864288330078125,0.0004962910221655478758274,NaN,None,None,NaN
2,bert,cehrbert_y_mort_1yr,23,0.3315093219280242919921875,0.0001769987222660016881661,NaN,None,None,NaN
4,bert,hibehrt_y_mort_1yr,17,0.3622503578662872314453125,0.0004318577345004255508762,NaN,None,None,NaN
12,bert,medbert_y_mort_1yr,24,0.3424382209777832031250000,0.0004508319885243493524776,NaN,None,None,NaN
7,big_bird,hparams-opt_y_mort_1yr,14,0.3263420462608337402343750,0.0002295841627830029409597,0.0032105036618538520740151,cls,True,NaN
1,descemb,bert-ft_y_mort_1yr,12,0.3398759067058563232421875,0.0000196158461954579062768,NaN,None,None,0.3000000000000000444089210
3,descemb,cls-ft_y_mort_1yr,18,0.3456993699073791503906250,0.0004937718211659636226990,NaN,None,None,0.3000000000000000444089210
11,genhpf,hparams-opt_y_mort_1yr,4,0.3321355879306793212890625,0.0000200863959543217700438,NaN,None,None,0.1000000000000000055511151
5,longformer,hparams-opt_y_mort_1yr,13,0.3286551535129547119140625,0.0004526773652751551373433,0.0010071785113063991962123,mean,False,NaN
10,modernbert,hparams-opt_y_mort_1yr,13,0.3221953511238098144531250,0.0002392565425637961922401,0.0028756040635929563421824,mean,True,NaN


In [11]:
# change this
study = df.iloc[1]
study

arch                     bert
task             behrt_y_mort
trial_number               18
value              0.18131164
learning_rate      0.00044618
weight_decay              NaN
pooling                  None
use_numeric              None
Name: 1, dtype: object

In [12]:
pd.read_csv('best_trials_summary.csv')

,trial_number,value,learning_rate,weight_decay,pooling,use_numeric,task,arch
0,12,0.18271545,0.00004994,0.00158814,cls,False,y_mort,bert
1,17,0.15056728,0.00004911,0.00318868,mean,False,y_icu_readmit_30,longformer
2,12,0.16771893,0.00004991,0.00280652,mean,True,y_mort,modernbert
3,2,0.24706557,0.00004225,0.00216827,cls,False,y_mort_1yr,big_bird
4,11,0.24682316,0.00004858,0.00115370,mean,False,y_mort_1yr,roformer
5,11,0.28047848,0.00004906,0.00111424,cls,False,y_los_7,big_bird
6,16,0.18286677,0.00004929,0.00381852,mean,False,y_mort,longformer
7,2,0.28599778,0.00003635,0.00746513,cls,False,y_los_7,modernbert
8,14,0.17949797,0.00004308,0.00248352,cls,True,y_mort,roberta
9,9,0.29298207,0.00003554,0.00162989,mean,False,y_los_7,longformer


In [16]:
import polars as pl
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")

polars.config.Config

In [18]:
pl.read_parquet('./downstream_idx.parquet').filter(pl.col('subject_id') == 10003502)#[:,:30]

subject_id,hadm_id,hosp_admission_time,hosp_discharge_time,icustay_id,icu_admission_time,icu_discharge_time,in_hosp_mort_time,out_mortality_time,n_events_hosp,n_events_icu,shard,hosp_los,hosp_los_hours,hosp_los_days,icu_los,icu_los_hours,icu_los_days,mort_24hr_offset,mort_48hr_offset,y_mort,y_mort_1yr,y_los_7,y_los_15,y_los_30,y_icu_readmit,y_icu_readmit_7,y_icu_readmit_15,y_icu_readmit_30,split,w24_min,w24_max,w48_min,w48_max,wStay_min,wStay_max,w24_start_512,w24_end_512,w24_start_1024,w24_end_1024,w24_start_1536,w24_end_1536,w48_start_512,w48_end_512,w48_start_1024,w48_end_1024,w48_start_1536,w48_end_1536,wStay_start_512,wStay_end_512,wStay_start_1024,wStay_end_1024,wStay_start_1536,wStay_end_1536
i64,i64,datetime[μs],datetime[μs],i64,datetime[μs],datetime[μs],datetime[μs],datetime[μs],u32,u32,str,duration[μs],f64,f64,duration[μs],f64,f64,datetime[μs],datetime[μs],i8,i8,i8,i8,i8,i8,i8,i8,i8,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
10003502,29011269,2169-08-26 16:14:00,2169-08-28 15:20:00,35796366,2169-08-26 21:30:32,2169-08-27 22:27:21,null,2169-09-10 00:00:00,893,809,"""9.parquet""",1d 23h 6m,47.1,1.9625,1d 56m 49s,24.946944444444444,1.0394560185185184,2169-08-27 21:30:32,2169-08-28 21:30:32,0,1,0,0,0,0,0,0,0,"""train""",1145,1980,1145,1987,1145,1987,1469,1980,1145,1980,1145,1980,1476,1987,1145,1987,1145,1987,1476,1987,1145,1987,1145,1987


In [1]:
import pandas as pd

In [2]:
pd.read_csv('/scratch/fs999/shamoutlab/data/mimic-iv-extracted/pretraining/test/10001884_episode1_timeseries.csv')

,Hours,Capillary refill rate,Diastolic blood pressure,Fraction inspired oxygen,Glascow coma scale eye opening,Glascow coma scale motor response,Glascow coma scale total,Glascow coma scale verbal response,Glucose,Heart Rate,Height,Mean blood pressure,Oxygen saturation,Respiratory rate,Systolic blood pressure,Temperature,Weight,pH
0,0.031944,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.081944,NaN,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.0,NaN,NaN,180.0,NaN,NaN,NaN
2,0.665278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,NaN,NaN,98.0,10.0,NaN,NaN,NaN,NaN
3,0.681944,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.0,NaN,NaN,167.0,NaN,NaN,NaN
4,1.665278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72.0,NaN,NaN,100.0,20.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
508,211.681944,NaN,83.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,102.0,NaN,NaN,147.0,NaN,NaN,NaN
509,212.665278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,87.0,NaN,NaN,68.0,9.0,NaN,NaN,NaN,NaN
510,213.665278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.0,NaN,NaN,45.0,8.0,NaN,NaN,NaN,NaN
511,215.665278,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,NaN,NaN,53.0,14.0,NaN,NaN,NaN,NaN
